In [5]:
import re
import fitz  # PyMuPDF
import pandas as pd
import streamlit as st
import io

def extract_text_from_pdf(pdf_file):
    """Trích xuất toàn bộ text từ file PDF (file object từ Streamlit)."""
    text = ""
    with fitz.open(stream=pdf_file.read(), filetype="pdf") as pdf:
        for page in pdf:
            text += page.get_text("text")
    return text

# Mở file cục bộ
with open(r"E:\D\HOCDIBANTRE\AnhSa\20251019_SGN_CHNLASC0483158_JP_PARTA_1.pdf", "rb") as f:
    text = extract_text_from_pdf(f)

print(text[:1000000])  # In thử 1000 ký tự đầu

 Official External#
X:\1.DOCUMENTS\2.OUTBOUND\2.DAILY_CLP\06-OCT CNC
CHNL
FVNFSXX - CONTAINER LOADING PLAN / CFS OUTBOUND RECEIPT
Booking Number :
ASC0481452
Master file:
1069668951
CLP No.:
J5101701
Vessel Name :
PANAY - 3CGIEN1NC
CY cut off:
15H30 15-OCT-2025
POD: 
TOKYO
Container number:
ECMU7438568
Tractor: 51C23089
ETD:
17/10/2025
Consignee:
JP
ETA:
26/10/2025
Seal number: 
M3834934
Drop-off:
TRANSIMEX
Loading date:
6-Oct-25
Cont size:
40' High Cube Dry
Tranship port:
TOKYO
Time from:
To:
*ECMU7438568*
DOOR NO.
Trucker: SOLOG
No.
Inbound 
Customs No.
Received Date
KN Reference
SP#
GTN S/O#
Shipper
Ship to Code
PO - LINE
CTN
 CBM 
Gross Weight
Quantity
SWB
1
307569290620
7/14/2025
1067986598
A10G69668951
VB25070105417654
CHANG SHIN VIETNAM CO., LTD. 1084
3503814055 - 00050
20
1.090
120.38
97
ASC0481452
2
307598384020
7/26/2025
1068244406
A10G69668951
VB25071509238103
CHANG SHIN VIETNAM COMPANY LIMITED
1084
3503829934 - 00070
17
0.670
62.28
66
ASC0481452
3
307596143210
7/23/2025
106

In [32]:
import re
import pandas as pd

def find_text_after_keyword(text, keyword, num_chars=50):
    """
    Tìm đoạn văn bản sau một keyword. 
    Nếu cùng dòng không có nội dung, lấy nội dung ở dòng kế tiếp.
    """
    # 1️⃣ Tìm trên cùng dòng
    pattern_same_line = re.escape(keyword) + r"[ \t]*(.{1," + str(num_chars) + r"})"
    match = re.search(pattern_same_line, text)
    if match:
        result = match.group(1).strip()
        if result:
            return result

    # 2️⃣ Nếu không có, tìm ở dòng kế tiếp
    pattern_next_line = re.escape(keyword) + r"\s*\n\s*(.{1," + str(num_chars) + r"})"
    match = re.search(pattern_next_line, text)
    if match:
        return match.group(1).strip()

    return None


def split_by_bill_names(text, bill_names):
    """
    Tách text thành các phần tương ứng với bill_names (theo thứ tự trong danh sách).
    Mỗi bill_name chỉ xuất hiện 1 lần, lấy nội dung từ bill_i đến bill_(i+1).
    """
    sections = []
    text_lower = text.lower()
    
    # Dò vị trí xuất hiện của từng bill_name trong text
    positions = []
    for b in bill_names:
        idx = text_lower.find(b.lower())
        if idx != -1:
            positions.append((idx, b))
    
    # Sắp xếp theo vị trí xuất hiện
    # positions.sort()
    
    # Tách nội dung giữa các bill
    for i, (start_idx, bill) in enumerate(positions):
        end_idx = positions[i+1][0] if i + 1 < len(positions) else len(text)
        content = text[start_idx + len(bill): end_idx].strip()
        sections.append((bill, content))
        
        text = text[end_idx:].strip()
    return sections
def split_by_bill_names(text, bill_names):
    """
    Cắt text thành các phần tương ứng với bill_names.
    - Sau khi lấy một section, loại bỏ phần đó khỏi text.
    - Không cho phép 2 bill liên tiếp giống nhau.
    """
    sections = []
    text_lower = text.lower()
    pattern = "|".join([re.escape(b.lower()) for b in bill_names])

    prev_bill = None
    while True:
        match = re.search(pattern, text_lower)
        if not match:
            break  # không còn bill nào

        bill = next(b for b in bill_names if b.lower() == match.group(0))

        # Bỏ qua nếu trùng với bill trước
        if bill == prev_bill:
            # Cắt bỏ phần trùng rồi tiếp tục tìm tiếp
            text = text[match.end():].strip()
            text_lower = text.lower()
            continue

        # Tìm bill tiếp theo (để biết điểm kết thúc)
        next_match = re.search(pattern, text_lower[match.end():])
        end_idx = match.end() + next_match.start() if next_match else len(text)
        content = text[match.end():end_idx].strip()

        # Lưu lại section
        sections.append((bill, content))

        # Cắt bỏ phần đã xử lý khỏi text
        text = text[end_idx:].strip()
        text_lower = text.lower()
        prev_bill = bill

    return sections
def extract_bill_sections(text, bill_names, keyword_dict, verbose=False):
    """
    Extract info for each bill section based on grouped keywords.
    Each main key in keyword_dict will become a column name.
    """
    sections = split_by_bill_names(text, bill_names)
    results = []

    for bill_name, content in sections:
        if verbose:
            print(f"Processing section: {bill_name}")
            print(f"Snippet: {content}...")
            print("-" * 40)

        entry = {"Bill Name": bill_name}
        for key, kw_list in keyword_dict.items():
            if not kw_list:
                entry[key] = "⚠️ Not Found"
                continue
            vals = None
            for kw in kw_list:
                cont = find_text_after_keyword(content, kw)
                if cont is not None:
                    vals = cont
            if vals:
                entry[key] = vals
            else:
                entry[key] = "⚠️ Not Found"

        results.append(entry)

    return pd.DataFrame(results)


bill_names = [
    "WAYBILL",
    "Trading Company  Commercial Invoice",
    "Factory Commercial Invoice",
    "Factory Packing List ",
    "MULTIPLE COUNTRY OF ORIGIN DECLARATION",
    "Japan Customs Form",
        "WAYBILL",
    "Trading Company  Commercial Invoice",
    "Factory Commercial Invoice",
    "Factory Packing List ",
    "MULTIPLE COUNTRY OF ORIGIN DECLARATION",
    "Japan Customs Form",
        "WAYBILL",
    "Trading Company  Commercial Invoice",
    "Factory Commercial Invoice",
    "Factory Packing List ",
    "MULTIPLE COUNTRY OF ORIGIN DECLARATION",
    "Japan Customs Form",
            "WAYBILL",
    "Trading Company  Commercial Invoice",
    "Factory Commercial Invoice",
    "Factory Packing List ",
    "MULTIPLE COUNTRY OF ORIGIN DECLARATION",
    "Japan Customs Form",
]


keywords = {
    "INV": ["Invoice#:", "Invoice Number:", "Invoice Number.:", "INVOICE NO."],
    "Total weight": ["Total Gross Kgs:", "Total Gross Weight:"],
    "total volume": [],
    "PO": ["PO-Item:", "P.O.#:", "PO#:", "PO Number:", "Reference PO#:"],
    "PO line": ["ITEM:", "PO Line Item Seq.#:", "PO Line#:", "Item Seq.:", "Sizes:"],
    "Style": ["Material:", "Material No:", "Material#:", "MATERIAL", "Material #:"],
    "total carton": ["Amount", "Cartons:"],
    "total quantity": ["Total Quantity:", "Total Qty:", "Qty:", "Quantity:"]
}

df = extract_bill_sections(text, bill_names, keywords, verbose=True)
for i in range(len(df)):
    if "-" in str(df.loc[i, "PO"]):
        parts = df.loc[i, "PO"].split("-")
        df.loc[i, "PO"] = parts[0]
        df.loc[i, "PO line"] = parts[1] if len(parts) > 1 else None
    df["PO"] = df["PO"].astype(str).str.split(",").str[0]
    df["PO line"] = df["PO line"].astype(str).str.split(",").str[0]
    df["total quantity"] = df["total quantity"].astype(str).str.split(" ").str[0]
df


Processing section: WAYBILL
Snippet: ...
----------------------------------------
Processing section: Trading Company  Commercial Invoice
Snippet: SELLER:
NIKE Global Trading BV Singapore Branch
30 Pasir Panjang Road
#10-31/32 Mapletree Business City
SINGAPORE 117440
SINGAPORE
Company Registration Number: T16FC0027D
GST Registration Number: M90367331L
BUYER:
NIKE JAPAN GROUP LLC
9-7-1 AKASAKA
MIDTOWN TOWER
TOKYO 107-6210
MINATO-KU
JAPAN
SHIP TO:
Nike Japan MN-Ichikawa
ACCA International-Digital/NSO
ESR ICHIKAWA Distribution Center 1F
678-55 Futamata
ICHIKAWA 272-0001
JAPAN
Invoice Number:
4302168583
Invoice Date:
July 30, 2025
Reference Invoice #:
PCVJ2509981
SAP Factory Code:
VP
Terms of Sale:
FOB
Mode of  Transportation:
Vessel
Vessel/Airline:
Ship Date:
August 06, 2025
From:
Ho Chi Minh, Vietnam
To:
Tokyo, Tk Japan
Shipping Marks
Description of Goods
Size Range
Quantity
Unit Price
Amount
31 Cartons of Footwear Division Goods
PR
JPY
1084
To:
Tokyo, Tk Japan
Country of Origin:
Vietnam

,Bill Name,INV,Total weight,total volume,PO,PO line,Style,total carton,total quantity
0,WAYBILL,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️
1,Trading Company Commercial Invoice,4302168583,52.49 Kgs,⚠️ Not Found,8501935073,5-11,HV9981-200,31,68
2,Factory Commercial Invoice,Invoice Date:,Total Net Weight:,⚠️ Not Found,3503842282,00090,Desc.:,31,Total
3,Factory Packing List,20250521,Total CBM:,⚠️ Not Found,8501935073,Customer Ship To: #:,Plant:,Total Units:,⚠️
4,MULTIPLE COUNTRY OF ORIGIN DECLARATION,PCVJ2509981,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,CONTENT 34% Plastic & 66% Textile,⚠️ Not Found,⚠️
5,Trading Company Commercial Invoice,4302168585,49.11 Kgs,⚠️ Not Found,8501935073,5-11,HV9981-600,27,64
6,Factory Commercial Invoice,Invoice Date:,Total Net Weight:,⚠️ Not Found,3503842282,00100,Desc.:,27,Total
7,Factory Packing List,20250521,Total CBM:,⚠️ Not Found,8501935073,Customer Ship To: #:,Plant:,Total Units:,⚠️
8,MULTIPLE COUNTRY OF ORIGIN DECLARATION,PCVJ2509987,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,CONTENT 34% Plastic & 66% Textile,⚠️ Not Found,⚠️
9,Trading Company Commercial Invoice,4302180885,78.53 Kgs,⚠️ Not Found,8501925841,5-12,IM8053-237,38,100


In [7]:
keywords = {
    "INV": ["Invoice#:", "Invoice Number:", "Invoice Number.:", "INVOICE NO."],
    "Total weight": ["Total Gross Kgs:"],
    "total volume": [],
    "PO": ["PO-Item:", "P.O.#:", "PO#:", "PO Number:"],
    "PO line": ["ITEM:", "PO Line No:", "PO Line#:"],
    "Style": ["Material:", "Material No:", "Material#:", "MATERIAL"],
    "total carton": ["Total Carton:", "Total Cartons:", "Total No. of Cartons:"],
    "total quantity": ["Total Quantity:", "Total Qty:", "Qty:"]
}

In [8]:
def find_text_after_keyword(text, keyword, num_chars=50):
    """
    Tìm đoạn văn bản sau một keyword. 
    Nếu cùng dòng không có nội dung, lấy nội dung ở dòng kế tiếp.
    """
    # 1️⃣ Tìm trên cùng dòng
    pattern_same_line = re.escape(keyword) + r"[ \t]*(.{1," + str(num_chars) + r"})"
    match = re.search(pattern_same_line, text)
    if match:
        result = match.group(1).strip()
        if result:
            return result

    # 2️⃣ Nếu không có, tìm ở dòng kế tiếp
    pattern_next_line = re.escape(keyword) + r"\s*\n\s*(.{1," + str(num_chars) + r"})"
    match = re.search(pattern_next_line, text)
    if match:
        return match.group(1).strip()

    return None

def split_by_bill_names(text, bill_names):
    """
    Tách text thành các phần tương ứng với bill_names (theo thứ tự trong danh sách).
    Mỗi bill_name chỉ xuất hiện 1 lần, lấy nội dung từ bill_i đến bill_(i+1).
    """
    sections = []
    text_lower = text.lower()
    
    # Dò vị trí xuất hiện của từng bill_name trong text
    positions = []
    for b in bill_names:
        idx = text_lower.find(b.lower())
        if idx != -1:
            positions.append((idx, b))
    
    # Sắp xếp theo vị trí xuất hiện
    positions.sort()
    
    # Tách nội dung giữa các bill
    for i, (start_idx, bill) in enumerate(positions):
        end_idx = positions[i+1][0] if i + 1 < len(positions) else len(text)
        content = text[start_idx + len(bill): end_idx].strip()
        sections.append((bill, content))
    
    return sections


def extract_bill_sections(text, bill_names, keywords):
    sections = split_by_bill_names(text, bill_names)
    results = []

    for bill_name, content in sections:
        print(f"Processing section: {bill_name}")
        print(f"Content snippet: {content}...")  # In thử 100 ký tự đầu của nội dung
        print("-" * 40)
        entry = {"Bill Name": bill_name}
        for kw in keywords:
            result = find_text_after_keyword(content, kw)
            entry[kw] = result if result else "⚠️ Not Found"
        results.append(entry)
    return pd.DataFrame(results)


bill_names = ["WAYBILL", "Trading Company  Commercial Invoice", "Factory Commercial Invoice", "Factory Packing List ", "MULTIPLE COUNTRY OF ORIGIN DECLARATION", "Japan Customs Form"]
    
keywords = [
    "Invoice#:",
    "PO-Item:",
    "Material:",
    "NO MARKS",
    "Reference Invoice #:",
    "Material#:",
    "Material #:",
    "Invoice Number:",
    "Total Gross Weight:",
    "Reference PO#:",
    "Total Number of Cartons: ",
    "Total Invoice Quantity:",
    "PO Line Item Seq. #:",
    "Invoice Number.:",
    "Material:",
    "Reference PO#:",
    '''INVOICE NO. 
    :''',
    "STYLE/CLR:"
    
]

df = extract_bill_sections(text, bill_names, keywords)

Processing section: WAYBILL
Content snippet: WAYBILL NUMBER
TOKYO,105-0014, JAPAN >
NUMBER OF ORIGINAL WAYBILLS
KGS
KGS
CBM
as agents for the carrier CMA CGM Asia Shipping Pte. Ltd.
SIGNED FOR THE CARRIER CMA CGM Asia Shipping Pte. Ltd.
BY
_______________________________________________
CMA CGM VIETNAM JSC
Said to contain
This Waybill is governed by the Terms and Conditions available on the CNC website (www.cnc-line.com) 
which the Merchant has read and accepted. The carrier is entitled to deliver the cargo to the Consignee, 
after payment of any outstanding Freight, on provision of proper proof of identity without the need to 
produce or surrender a copy of this Sea Waybill.
Shippers stow, load and count
Demurrage and detention payable by the Merchant as per the Carrier’s tariff available on the web site 
www.cnc-line.com, or in any of his agency. However if special free time conditions are granted, then rates 
applicable as per general tariff grid shall start from the day following t

In [9]:
df

,Bill Name,Invoice#:,PO-Item:,Material:,NO MARKS,Reference Invoice #:,Material#:,Material #:,Invoice Number:,Total Gross Weight:,Reference PO#:,Total Number of Cartons:,Total Invoice Quantity:,PO Line Item Seq. #:,Invoice Number.:,INVOICE NO. \n :,STYLE/CLR:
0,WAYBILL,PCVJ2508881,"3503830020-20, Customer PO:","HV9981-600, Name: W NIKE FLEX TRAIN,",⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found
1,Trading Company Commercial Invoice,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,PCVJ2509981,HV9981-200,⚠️ Not Found,4302168583,52.49 Kgs,8501935073,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found
2,Factory Commercial Invoice,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,"Tokyo, Tk Japan",⚠️ Not Found,Desc.:,Invoice Date:,Total Net Weight:,3503842282,⚠️ Not Found,Total Amount:,Reference PO#:,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found
3,Factory Packing List,⚠️ Not Found,⚠️ Not Found,Plant:,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,8501935073,⚠️ Not Found,⚠️ Not Found,⚠️ Not Found,20250521,⚠️ Not Found,⚠️ Not Found
4,MULTIPLE COUNTRY OF ORIGIN DECLARATION,⚠️ Not Found,⚠️ Not Found,Plant:,⚠️ Not Found,PCVJ2509987,HV9981-600,Desc.:,4302168585,49.11 Kgs,8501935073,⚠️ Not Found,Total Amount:,Reference PO#:,20250521,⚠️ Not Found,⚠️ Not Found
